# Deep JSCC — Tutorial Demo

End-to-end Deep Joint Source-Channel Coding on CIFAR-10, with two stories:

1. **Channel noise and graceful degradation** — Deep JSCC vs. raw transmission.
2. **Rate-distortion trade-off** — how much bandwidth `k/n` do you actually need?

**Before running:** *Runtime → Change runtime type → T4 GPU* (free).

## 0. Setup

In [ ]:
!git clone https://github.com/<YOUR_GITHUB_USER>/deep-jscc-demo.git
%cd deep-jscc-demo
!pip install -q -r requirements.txt

## 1. Story 1 — Channel noise and graceful degradation

Train two fixed-SNR models (`SNR_train ∈ {1, 7}` dB), then sweep test SNR. ~10 min on T4.

In [ ]:
!python train.py        # fixed SNRs: {1, 7} dB

In [ ]:
!python eval.py         # PSNR/SSIM curves + reconstruction grid

In [ ]:
from IPython.display import Image, display
display(Image('results/awgn_snr_sweep.png'))
display(Image('results/awgn_reconstructions.png'))

**Reading the figures:**
- The grid alternates **Raw @ X dB** (pixels through the channel, no encoder) with **JSCC @ X dB**. At every SNR, Raw is pure snow; JSCC stays recognizable. This is the elevator pitch for semantic communication.
- The PSNR/SSIM curves show **graceful degradation** — no cliff at low SNR, just a smooth quality drop. That's what classical separation-based coding cannot do.

## 2. Story 2 — Rate-distortion: how much bandwidth do we need?

Train at a single SNR (7 dB) across three latent sizes, i.e. three **bandwidth ratios `k/n`**, then plot quality vs. bandwidth.

| `latent_ch` |  k/n  |
|------------:|:-----:|
|           4 | 1/12  |
|           8 | 1/6   |
|          16 | 1/3   |

In [ ]:
!python train.py --snr_train_list 7 --latent_ch_list 4 8 16

In [ ]:
!python eval.py --mode bw_sweep --bw_train_snr 7dB

In [ ]:
display(Image('results/awgn_bw_sweep.png'))
display(Image('results/awgn_bw_reconstructions.png'))

**Reading the figures:**
- Quality saturates as `k/n` grows — diminishing returns. The knee tells you the cheapest bandwidth you can get away with.
- The reconstruction grid shows the same image at each bandwidth — the drop in detail from `k/n=1/3` to `k/n=1/12` is visible.

## 3. Optional — Rayleigh fading

Repeat Story 1 over a Rayleigh-fading channel instead of AWGN.

In [ ]:
# !python train.py --channel rayleigh
# !python eval.py  --channel rayleigh
# display(Image('results/rayleigh_snr_sweep.png'))